# Inventory Optimization 

In [1]:
import pandas as pd

In [2]:
Auto = pd.read_csv('/Users/Stalwarts/Desktop/Personal/AI/ecommerce_dataset.csv', on_bad_lines='skip')

In [3]:
Auto.head()

,order_id,order_date,order_year,order_month,order_day,order_hour,order_minute,order_second,is_weekend,order_status,...,campaign_source,device_type,traffic_source,session_duration_minutes,pages_visited,abandoned_cart_before,fraud_risk_score,profit_margin_percent,order_priority,support_ticket_created
0,ORD-XAJI0,2024-10-14 11:20:05.496679,2024,10,14,11,20,5,No,Completed,...,Email,Mobile,Search,52.1,13,No,45.3,28.66,Medium,Yes
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,2024,10,21,0,49,44,No,Completed,...,Google Ads,Desktop,Referral,36.2,15,No,97.1,33.91,Low,Yes
2,ORD-YTJXE,2025-03-17 19:49:36.983317,2025,3,17,19,49,36,No,Completed,...,Facebook,Tablet,Search,43.1,13,No,43.8,52.77,High,No
3,ORD-EIMVI,2024-09-27 06:24:44.913768,2024,9,27,6,24,44,No,Completed,...,Instagram,Mobile,Direct,39.5,18,Yes,96.9,38.15,Low,Yes
4,ORD-OR56F,2025-05-21 17:10:48.401882,2025,5,21,17,10,48,No,Completed,...,Facebook,Tablet,Social,28.0,3,Yes,45.8,33.05,High,Yes


In [4]:
Auto.columns

Index(['order_id', 'order_date', 'order_year', 'order_month', 'order_day',
       'order_hour', 'order_minute', 'order_second', 'is_weekend',
       'order_status', 'return_reason', 'customer_id', 'customer_name',
       'gender', 'age', 'customer_segment', 'country', 'city',
       'customer_loyalty_score', 'total_orders_by_customer',
       'account_creation_date', 'product_id', 'product_name', 'category',
       'sub_category', 'brand', 'product_rating_avg', 'product_reviews_count',
       'stock_quantity', 'unit_price_usd', 'quantity', 'discount_percent',
       'discount_amount_usd', 'total_price_usd', 'cost_usd', 'profit_usd',
       'tax_usd', 'currency', 'payment_method', 'payment_status',
       'installment_plan', 'shipping_method', 'shipping_cost_usd',
       'delivery_days', 'shipping_country', 'warehouse_location',
       'delivery_status', 'rating', 'review_sentiment', 'customer_feedback',
       'coupon_used', 'coupon_code', 'campaign_source', 'device_type',
       'traf

### 1. Stock-out Prediction

In [5]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (mean_absolute_error,classification_report,confusion_matrix)

In [6]:
con = duckdb.connect("inventory.duckdb", read_only = True)

df = con.execute("""
SELECT *
FROM read_csv_auto('/Users/Stalwarts/Desktop/Personal/AI/ecommerce_dataset.csv')
""").df()
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
query = """
SELECT
    product_id,
    product_name,
    order_date,

    category,
    sub_category,
    brand,

    AVG(unit_price_usd) AS avg_price,
    AVG(discount_percent) AS avg_discount,

    SUM(quantity) AS daily_demand,

    AVG(stock_quantity) AS avg_stock,

    AVG(product_rating_avg) AS avg_rating,

    COUNT(DISTINCT order_id) AS num_orders,

    MAX(is_weekend) AS is_weekend

FROM df

GROUP BY
    product_id,
    product_name,
    order_date,
    category,
    sub_category,
    brand
"""

daily_df = duckdb.query(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
print(daily_df)

        product_id            product_name                 order_date  \
0         PRD-FSRL  Coffee Maker Automatic 2025-07-06 03:35:47.472580   
1         PRD-LD2J        Swimming Goggles 2024-05-14 02:49:55.022824   
2         PRD-27WX          USB Hub 7-Port 2024-02-11 10:56:12.441825   
3         PRD-I6EN     LED Strip Lights 5m 2024-06-25 04:47:50.466358   
4         PRD-SAUM          USB Hub 7-Port 2025-08-03 15:34:41.535573   
...            ...                     ...                        ...   
1000118   PRD-UPI7    Resistance Bands Set 2025-06-17 10:49:30.027806   
1000119   PRD-Z3SG    Denim Jeans Slim Fit 2025-01-02 11:42:16.679002   
1000120   PRD-ZR1L           Exercise Ball 2025-10-17 21:31:10.831655   
1000121   PRD-MKJU          USB Hub 7-Port 2025-08-07 19:11:22.187411   
1000122   PRD-YML2    Resistance Bands Set 2024-07-27 15:48:16.443955   

            category     sub_category     brand  avg_price  avg_discount  \
0               Home          Kitchen     GoPro

In [9]:
daily_df = daily_df.sort_values(
    ['product_id', 'order_date']
)

# Lag Features
daily_df['lag_1'] = (
    daily_df.groupby('product_id')['daily_demand']
    .shift(1)
)

daily_df['lag_7'] = (
    daily_df.groupby('product_id')['daily_demand']
    .shift(7)
)

# Rolling Features
daily_df['rolling_avg_7'] = (
    daily_df.groupby('product_id')['daily_demand']
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=1
        ).mean()
    )
)

daily_df['rolling_avg_30'] = (
    daily_df.groupby('product_id')['daily_demand']
    .transform(
        lambda x: x.rolling(
            window=30,
            min_periods=1
        ).mean()
    )
)

In [10]:
daily_df['lag_1'] = daily_df['lag_1'].fillna(0)

daily_df['lag_7'] = daily_df['lag_7'].fillna(
    daily_df['rolling_avg_7']
)

daily_df['rolling_avg_7'] = (
    daily_df['rolling_avg_7']
    .fillna(daily_df['daily_demand'])
)

daily_df['rolling_avg_30'] = (
    daily_df['rolling_avg_30']
    .fillna(daily_df['rolling_avg_7'])
)

# Calendar Features
daily_df['month'] = daily_df['order_date'].dt.month
daily_df['day_of_week'] = daily_df['order_date'].dt.dayofweek

In [11]:
daily_df['stock_out'] = np.where(
    daily_df['avg_stock'] <= 2,
    1,
    0
)

In [12]:
features = [
    'avg_price',
    'avg_discount',
    'avg_rating',
    'num_orders',
    'avg_stock',

    'lag_1',
    'lag_7',
    'rolling_avg_7',
    'rolling_avg_30',

    'month',
    'day_of_week',

    'category',
    'sub_category',
    'product_name',
    'brand',
    'is_weekend'
]

In [13]:
print(daily_df.columns.tolist())

['product_id', 'product_name', 'order_date', 'category', 'sub_category', 'brand', 'avg_price', 'avg_discount', 'daily_demand', 'avg_stock', 'avg_rating', 'num_orders', 'is_weekend', 'lag_1', 'lag_7', 'rolling_avg_7', 'rolling_avg_30', 'month', 'day_of_week', 'stock_out']


In [14]:
X = daily_df[features]

y_demand = daily_df['daily_demand']


In [15]:
numeric_features = [
    'avg_price',
    'avg_discount',
    'avg_rating',
    'num_orders',
    'avg_stock',
    'lag_1',
    'lag_7',
    'rolling_avg_7',
    'rolling_avg_30',
    'month',
    'day_of_week'
]

categorical_features = [
    'category',
    'sub_category',
    'product_name',
    'brand',
    'is_weekend'
]

In [16]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

#### Demand Forecasting Model

In [17]:
demand_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', PoissonRegressor(alpha=0.1, max_iter=500))
])

In [18]:
train_df = daily_df[daily_df['order_date'] < '2025-01-01']
test_df = daily_df[daily_df['order_date'] >= '2025-01-01']

X_train = train_df[features]
X_test = test_df[features]

y_train_demand = train_df['daily_demand']
y_test_demand = test_df['daily_demand']

In [19]:
demand_model.fit(X_train, y_train_demand)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [20]:
demand_predictions = demand_model.predict(X_test)

print(
    "Demand MAE:",
    mean_absolute_error(y_test_demand, demand_predictions)
)

Demand MAE: 0.4696676230741604


In [21]:
forecast_df = test_df.copy()

forecast_df['predicted_demand'] = np.round(
    demand_predictions,
    2
)

In [22]:
forecast_df['days_until_stockout'] = np.where(
    forecast_df['predicted_demand'] > 0,

    forecast_df['avg_stock'] /
    forecast_df['predicted_demand'],

    np.nan
)

In [23]:
forecast_df['demand_stddev'] = (
    forecast_df.groupby('product_id')['daily_demand']
    .transform(
        lambda x: x.rolling(
            window=30,
            min_periods=1
        ).std()
    )
)

forecast_df['demand_stddev'] = (
    forecast_df['demand_stddev']
    .fillna(0)
)

In [24]:
Z_SCORE = 1.65

LEAD_TIME_DAYS = 7

forecast_df['safety_stock'] = (
    Z_SCORE *
    forecast_df['demand_stddev'] *
    np.sqrt(LEAD_TIME_DAYS)
)

In [25]:
final_forecast_df = forecast_df[[
    'order_date',

    'product_id',
    'product_name',
    'category',
    'sub_category',
    'brand',

    'avg_stock',

    'daily_demand',
    'predicted_demand',

    'safety_stock',

    'avg_price',
    'avg_discount',

    'rolling_avg_7',
    'rolling_avg_30'
]]

In [26]:
final_forecast_df = final_forecast_df.rename(columns={

    'avg_stock': 'current_stock',

    'daily_demand': 'actual_demand',

    'avg_price': 'unit_price',

    'avg_discount': 'discount_percent'
})

In [27]:
final_forecast_df.head(20)

,order_date,product_id,product_name,category,sub_category,brand,current_stock,actual_demand,predicted_demand,safety_stock,unit_price,discount_percent,rolling_avg_7,rolling_avg_30
973472,2025-01-04 11:19:07.731609,PRD-0000,Water Bottle Insulated,Sports,Sports Wear,JBL,646.0,2.0,2.51,0.000000,248.92,0.0,3.000000,3.000000
897252,2025-08-22 19:33:28.471487,PRD-0000,USB-C Fast Charger 65W,Electronics,Accessories,Adidas,894.0,3.0,2.63,3.086867,285.29,0.0,3.000000,3.000000
441869,2025-09-30 20:59:56.515064,PRD-0001,Hooded Sweatshirt,Clothing,Shoes,Huawei,74.0,5.0,5.41,0.000000,19.86,5.0,5.000000,5.000000
538737,2025-04-08 14:10:43.246409,PRD-0003,Storage Box Set,Home,Decor,Adidas,737.0,2.0,1.90,0.000000,38.85,5.0,2.000000,2.000000
307736,2025-04-02 07:09:37.397213,PRD-0008,Foam Roller,Health,Supplements,Lenovo,962.0,1.0,1.97,0.000000,145.11,10.0,2.333333,2.333333
760659,2025-09-19 07:49:17.123309,PRD-0009,Swimming Goggles,Sports,Outdoor,Apple,11.0,5.0,2.68,0.000000,230.47,0.0,3.000000,3.000000
162403,2025-09-11 11:07:57.130136,PRD-000D,Jump Rope Speed,Sports,Sports Wear,JBL,222.0,2.0,1.87,0.000000,146.57,5.0,2.000000,2.000000
562661,2025-10-11 14:15:43.005182,PRD-000E,Casual Sneakers,Clothing,Mens Wear,Dell,525.0,4.0,3.84,0.000000,42.74,0.0,4.000000,4.000000
464641,2025-12-25 22:12:37.170449,PRD-000F,Desk Lamp LED,Home,Furniture,Adidas,147.0,3.0,3.05,0.000000,33.09,10.0,3.500000,3.500000
443845,2026-01-24 09:38:02.987364,PRD-000F,Fitness Smart Band,Health,Fitness,Nikon,133.0,1.0,2.27,6.173735,36.32,10.0,2.666667,2.666667


#### Stock-out Prediction Model

In [28]:
stockout_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ))
])

In [29]:
y_train_stockout = train_df['stock_out']
y_test_stockout = test_df['stock_out']

stockout_model.fit(X_train, y_train_stockout)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [30]:
stockout_predictions = stockout_model.predict(X_test)

In [31]:
print(classification_report(
    y_test_stockout,
    stockout_predictions
))

print(confusion_matrix(
    y_test_stockout,
    stockout_predictions
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    543251
           1       1.00      1.00      1.00      1623

    accuracy                           1.00    544874
   macro avg       1.00      1.00      1.00    544874
weighted avg       1.00      1.00      1.00    544874

[[543251      0]
 [     0   1623]]


In [32]:
print(
    "Stockout MAE:",
    mean_absolute_error(y_test_stockout, stockout_predictions)
)

Stockout MAE: 0.0


In [33]:
stockout_probabilities = stockout_model.predict_proba(X_test)[:, 1]

In [34]:
forecast_df['stockout_probability'] = np.round(
    stockout_probabilities,
    3
)

In [35]:
forecast_df['days_until_stockout'] = np.where(
    forecast_df['predicted_demand'] > 0,

    forecast_df['avg_stock'] /
    forecast_df['predicted_demand'],

    np.nan
)

In [36]:
forecast_df['reorder_point'] = (
    forecast_df['predicted_demand']
    *
    LEAD_TIME_DAYS
) + forecast_df['safety_stock']

In [37]:
forecast_df['current_stock'] = (
    forecast_df['avg_stock']
)

In [38]:
forecast_df['reorder_quantity'] = np.where(

    forecast_df['current_stock']
    <
    forecast_df['reorder_point'],

    np.ceil(
        forecast_df['reorder_point']
        -
        forecast_df['current_stock']
    ),

    0
)

In [39]:
reorder_df = forecast_df[
    forecast_df['reorder_quantity'] > 0
]

In [40]:
print(
    reorder_df[[
        'product_id',
        'product_name',
        'current_stock',
        'predicted_demand',
        'safety_stock',
        'reorder_point',
        'reorder_quantity'
    ]].head(20)
)

       product_id                     product_name  current_stock  \
760659   PRD-0009                 Swimming Goggles           11.0   
520690   PRD-001I              LED Strip Lights 5m           15.0   
371764   PRD-006F                   USB Hub 7-Port            3.0   
559127   PRD-0075                 Portable SSD 1TB           21.0   
433087   PRD-0095                Treadmill Folding            0.0   
33860    PRD-00AN           Sports Bra High Impact           18.0   
842103   PRD-00C5           USB-C Fast Charger 65W            1.0   
130440   PRD-00HJ   Digital Blood Pressure Monitor           14.0   
258476   PRD-00KD                     Baseball Cap            1.0   
505146   PRD-00KN  Screen Protector Tempered Glass           11.0   
882249   PRD-00OY              Digital Thermometer           16.0   
900899   PRD-00PD                    4K Webcam Pro           13.0   
261134   PRD-00QS                    Desk Lamp LED           11.0   
505903   PRD-00V2                Y

In [41]:
final_forecast_df = forecast_df[[
    'order_date',

    'product_id',
    'product_name',
    'category',
    'sub_category',
    'brand',

    'avg_stock',

    'daily_demand',
    'predicted_demand',

    'avg_price',
    'avg_discount',

    'rolling_avg_7',
    'rolling_avg_30',

    'stockout_probability',

    'days_until_stockout',

    'safety_stock',
    'reorder_quantity',
]]

In [42]:
final_forecast_df = final_forecast_df.sort_values(
    by='stockout_probability',
    ascending=False
)

In [43]:
final_forecast_df.head(20)

,order_date,product_id,product_name,category,sub_category,brand,avg_stock,daily_demand,predicted_demand,avg_price,avg_discount,rolling_avg_7,rolling_avg_30,stockout_probability,days_until_stockout,safety_stock,reorder_quantity
349260,2025-09-06 22:37:14.161244,PRD-JWX7,Wireless Mouse 2.4GHz,Electronics,Smartphones,Samsung,0.0,4.0,3.83,53.02,5.0,4.000000,4.000000,0.791,0.000000,0.000000,27.0
438214,2025-01-26 19:21:15.174842,PRD-GOE7,Treadmill Folding,Sports,Outdoor,Adidas,1.0,5.0,5.33,163.16,5.0,5.000000,5.000000,0.781,0.187617,0.000000,37.0
24996,2025-06-01 04:15:10.923362,PRD-OIF8,USB-C Fast Charger 65W,Electronics,Smartphones,Samsung,0.0,5.0,4.25,491.85,0.0,4.500000,4.500000,0.771,0.000000,3.086867,33.0
580397,2025-10-25 06:41:01.502426,PRD-F1FA,Storage Box Set,Home,Kitchen,Anker,2.0,4.0,3.60,249.21,0.0,4.000000,4.000000,0.761,0.555556,0.000000,24.0
119758,2025-02-23 18:21:26.294055,PRD-2M47,Bedsheet Set Cotton,Home,Kitchen,GoPro,1.0,4.0,3.77,221.95,10.0,4.000000,4.000000,0.761,0.265252,0.000000,26.0
592535,2025-09-13 00:47:43.500684,PRD-01CA,Desk Lamp LED,Home,Bedding,Sony,2.0,5.0,5.42,215.73,0.0,5.000000,5.000000,0.761,0.369004,0.000000,36.0
701237,2025-09-28 18:40:10.099523,PRD-C4J9,Air Fryer 5L,Home,Furniture,Asus,1.0,5.0,5.41,211.65,5.0,5.000000,5.000000,0.760,0.184843,0.000000,37.0
836578,2025-01-25 21:04:14.697438,PRD-5ZM0,Coffee Maker Automatic,Home,Decor,Adidas,0.0,4.0,3.77,109.15,5.0,4.000000,4.000000,0.752,0.000000,0.000000,27.0
616372,2025-08-17 20:38:14.865668,PRD-IMWN,Massage Gun Deep Tissue,Health,Supplements,Huawei,1.0,5.0,3.65,17.80,5.0,4.000000,4.000000,0.751,0.273973,0.000000,25.0
87803,2025-11-22 19:29:22.190646,PRD-EE85,Vacuum Robot Cleaner,Home,Furniture,JBL,0.0,4.0,3.85,192.66,10.0,4.000000,4.000000,0.751,0.000000,0.000000,27.0


In [44]:
len(final_forecast_df)

544874

In [45]:
final_forecast_df.to_csv(
    'inventory_forecast_output.csv',
    index=False
)